# 05 · Multilingual OCR with OpenEnv

Stream a small window of CognitiveLab's Nayana corpus, derive full-page OCR, section OCR, and multiple-choice VQA tasks, and replay the same task across independent RL rollouts.

This notebook calls the package and training runner in this repository. The CPU walkthrough is runnable without a GPU. GPU training is optional and disabled by default. The eight-page smoke window is not a held-out benchmark.

From `05-multilingual-ocr/`, start a kernel with the package installed:

```bash
uv run --frozen --project envs/nayana_ocr --extra train --with jupyterlab jupyter lab
```

Dataset: [Cognitive-Lab/NayanaOCR_Corpus_2025](https://huggingface.co/datasets/Cognitive-Lab/NayanaOCR_Corpus_2025), CC BY-NC 4.0. Prepared images, annotations, and crops retain this attribution and license.


In [ ]:
import json
from pathlib import Path

PROJECT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "05-multilingual-ocr")
    if candidate.name == "05-multilingual-ocr" and (candidate / "project.yaml").exists()
)
SNAPSHOT = PROJECT / "data" / "snapshots" / "real-smoke-v2"
print(PROJECT)

## Prepare a bounded real-data window

The source revision and Datasets version are pinned. Preparation streams Parquet sequentially with image decoding disabled, then writes only selected pages and derived crops to local storage. Rerun this cell to resume the same preparation after interruption.


In [ ]:
from nayana_ocr.data.catalog import Catalog
from nayana_ocr.data.prepare import PrepareConfig, prepare

config = PrepareConfig(languages=("en", "kn", "hi", "ar"), pages_per_language=2)
manifest = prepare(SNAPSHOT, config)
print(
    json.dumps(
        {
            key: manifest[key]
            for key in ("snapshot_id", "pages", "counts", "media_bytes")
        },
        indent=2,
    )
)

## Inspect tasks without decoding the dataset

These counts describe this prepared window, not the full million-page corpus. The first two pages per language in the recorded smoke all belong to train. All translations and pages from the same document follow the same partition.


In [ ]:
catalog = Catalog(SNAPSHOT)
for split in ("train", "validation", "test"):
    print(split, catalog.count(split))
tasks = catalog.task_range("train", 0, min(100, catalog.count("train")))
print(
    [
        {key: task[key] for key in ("task_id", "language", "family", "page_id")}
        for task in tasks[:4]
    ]
)
assert all("reference" not in task for task in tasks)

## Replay a full-page OCR task through OpenEnv

Discovery returns metadata. A WebSocket reset selects an immutable task ID and the binary asset endpoint returns the image. The image cache verifies its SHA-256. A second session receives the same task, with separate episode state. Full-page OCR preserves the canvas but masks unannotated areas; its reference joins valid, non-overlapping regions in geometric reading order. Arabic columns run right to left. Section OCR and MCQ VQA share the same catalog.


In [ ]:
from IPython.display import display
from nayana_ocr.client import connect
from nayana_ocr.models import NayanaAction
from nayana_ocr.runtime import local_server
from nayana_ocr.training import AssetCache

with local_server(SNAPSHOT) as url:
    cache = AssetCache(url)
    with connect(url) as first, connect(url) as second:
        task = next(
            t for t in tasks if t["language"] == "kn" and t["family"] == "page_ocr"
        )
        observation = first.reset(task_id=task["task_id"]).observation
        repeated = second.reset(task_id=task["task_id"]).observation
        assert observation.asset_sha256 == repeated.asset_sha256
        print(observation.prompt)
        image = cache.image(observation)
        display(image)
        image.close()
        result = first.step(NayanaAction(answer=""))
        print("Empty-answer reward:", result.reward, result.observation.metrics)

## Run the complete CPU integration check

The trusted test driver can read local references to check perfect scoring. References are absent from OpenEnv discovery, observations, and terminal responses. The separate Gradio playground intentionally reveals the reference after scoring. This checks transport and reward correctness; no model generates answers in this cell.


In [ ]:
from nayana_ocr.smoke import probe

with local_server(SNAPSHOT) as url:
    verification = probe(url, catalog)
print(json.dumps(verification, indent=2))

## Optional GPU training

Training uses the same `train/grpo_nayana.py` script as HF Jobs. First prepare a document-diverse window with train and test examples for every requested language/task group. The runner checks this before downloading model weights. If coverage is missing, prepare more documents in a new directory.

The default mix balances language × task groups. TRL receives small task-ID records; its sampler repeats each ID for all completions in a GRPO group. Images stay in the environment and are fetched by the shared cache. The processor applies an explicit image budget; the server keeps original crop resolution.

The following cell is disabled until you set `RUN_TRAINING = True`. It requires CUDA and may download a larger data window and model weights.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    import sys

    training_snapshot = PROJECT / "data" / "snapshots" / "train-256-v2"
    prepare(
        training_snapshot,
        PrepareConfig(pages_per_language=256, max_media_bytes=8_000_000_000),
    )
    subprocess.run(
        [
            sys.executable,
            str(PROJECT / "train" / "grpo_nayana.py"),
            "--snapshot",
            str(training_snapshot),
            "--smoke",
            "--output-dir",
            str(PROJECT / "results" / "local-notebook-smoke"),
        ],
        check=True,
    )
else:
    print("GPU training disabled. CPU data and environment checks above are complete.")

## HF Jobs and Spaces

Use `train/hf_job.py` with a full pushed Git commit, so a job runs the reviewed source and frozen lockfile. CPU environment smoke, real-data smoke, and GPU training are separate modes. `train/deploy_space.py` bundles a chosen prepared window into an explicitly named Docker Space.

[REPRODUCE.md](../REPRODUCE.md) contains complete commands, scoring details, preparation limits, and checkpoint caveats. No Space, collection, model, or job is created by the default notebook walkthrough.
